# **Caculate RFF Feature Based on the Preamble**
Add v2.0 feature computation, corrected L-LTF handlin_20261025_by_Reitaku_JZ
See CHANGELOG.md for details.

Fix the equalisation pipeline so that stored features describe the
received symbols directly. Accuracy on 123 devices rises from 82.17%
to 88.64% without Kalman filtering, and from 88.04% to 94.71% with it.

1. Remove the residual subtraction after equalisation.
2. Restore the magnitude error sign to |Y| - |L|.
3. Rewrite the phase error as angle(Y * conj(L)).
4. Restrict statistics to the 52 active subcarriers.
5. Move the fractal dimension to the 288-sample time-domain preamble.
6. Drop the duplicate FFT.
7. Remove a dead comparison.
8. Keep iq_preamble, now holding the equalised symbols.
9. Add the four-panel feature display cell.
10. Reduce the column count from 23 to 22.

This section extracts ten representative radio-frequency fingerprinting (RFF) features from IEEE 802.11 preamble data. The extracted features include the raw preamble samples, long-preamble I/Q samples, multiple CFO estimations (total, STS-based, and LTS-based), phase and magnitude error metrics, I/Q gain imbalance, and fractal-dimension-based descriptors.

In [ ]:
import os
import math
import ast
import re
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------- Utility Functions -----------------------------

def distance(a: float, b: float, c: float, d: float) -> float:
    """Euclidean distance between (a,b) and (c,d)."""
    return math.hypot(a - c, d - b)

def power(x: np.ndarray) -> float:
    """Sum of squares."""
    x = np.asarray(x, dtype=float)
    return float(np.sum(x * x))

def frac_dimension(sample_real: np.ndarray, sample_imag: np.ndarray, tau: int) -> float:
    """Fractal dimension estimator (same formula as MATLAB version)."""
    N = len(sample_real)
    if N < 2 or tau < 2:
        return float("nan")

    # sum over step = 1 (adjacent samples)
    m1_end = N - (N % tau) - 2
    if m1_end < 0:
        return float("nan")
    summary_1 = 0.0
    for i in range(0, m1_end + 1):
        summary_1 += distance(sample_real[i], sample_imag[i],
                              sample_real[i + 1], sample_imag[i + 1])

    # sum over step = tau
    m_tau_end = (N - (N % tau)) // tau - 2
    if m_tau_end < 0:
        return float("nan")
    summary_tau = 0.0
    for i in range(0, m_tau_end + 1):
        summary_tau += distance(sample_real[i * tau], sample_imag[i * tau],
                                sample_real[(i + 1) * tau], sample_imag[(i + 1) * tau])

    if summary_1 <= 0 or summary_tau <= 0:
        return float("nan")

    dim = 0.5 * (3.0 - ((math.log10(summary_tau) - math.log10(summary_1)) / math.log10(tau)))
    return dim

def parse_complex_list(cell) -> np.ndarray:

    if pd.isna(cell):
        return np.array([], dtype=complex)

    s = str(cell)
    # Remove all whitespace characters (including spaces, newlines, and tabs).
    s = re.sub(r'\s+', '', s)
    s = s.replace('i', 'j')

    if not (s.startswith('[') and s.endswith(']')):
        s = f'[{s}]'

    # Use regular expressions to extract each complex number of the form a±bj, where a/b can include decimals or scientific notation.
    patt = re.compile(
        r'[+-]?'
        r'(?:\d+(?:\.\d+)?(?:e[+-]?\d+)?)'   # real part
        r'[+-]'
        r'(?:\d+(?:\.\d+)?(?:e[+-]?\d+)?)j', # Imaginary part
        re.IGNORECASE
    )
    parts = patt.findall(s)
    if parts:
        try:
            return np.array([complex(p) for p in parts], dtype=complex)
        except Exception:
            pass  

    try:
        lst = ast.literal_eval(s)
        return np.array(lst, dtype=complex)
    except Exception:
        return np.array([], dtype=complex)  

# [改6] iq_preamble 保留，序列化函数随之恢复（原 [删3] 撤回）
def fmt_complex_list(arr: np.ndarray) -> str:

    out = []
    for z in np.asarray(arr, dtype=complex).tolist():
        a = z.real
        b = z.imag
        if abs(a) < 1e-12: a = 0.0
        if abs(b) < 1e-12: b = 0.0
        sign = '+' if b >= 0 else ''
        out.append(f"{a:g}{sign}{b:g}j")
    return '[' + ', '.join(out) + ']'

# ---------------------- Main Processing -------------------------------

def main():
    # 参数
    N_target = 1000      # 每个文件处理前 N 行
    tau = 3              # fractal dimension 参数

    # 路径
    try:
        script_dir = Path(__file__).resolve().parent
    except NameError:
        script_dir = Path(os.getcwd()).resolve()

    targetFolderName = '/kaggle/input/rffi-123-m5stack-iq-wifi-802-11g-2-4g'
    storeFolderName  = '/kaggle/working/RFF_DataSet_with_feature'

    src_dir = script_dir / targetFolderName
    dst_dir = script_dir / storeFolderName
    dst_dir.mkdir(parents=True, exist_ok=True)

    # List files (excluding hidden files)
    all_files = sorted([p for p in src_dir.iterdir() if p.is_file() and not p.name.startswith('.')])
    error_files = []
    ok_files = []

    max_count = min(123, len(all_files))
    for file_idx in range(max_count):
        src_path = all_files[file_idx]
        dst_path = dst_dir / src_path.name

        try:
            df = pd.read_csv(src_path)
        except Exception as e:
            print(f"[READ-ERR] {src_path.name}: {e}")
            error_files.append(src_path.name)
            continue

        columnName = 'preamble'
        if columnName not in df.columns:
            print(f'Column "{columnName}" does not exist in CSV file: {src_path.name}')
            error_files.append(src_path.name)
            continue

       
        H = len(df)
        # [改6] iq_preamble 保留。存的是均衡后的两段频域符号，不再减理想值
        iq_preamble_col = [None] * H
        frac_dim_col    = [None] * H
        # [改5] 分形维数只保留一列
        # frac_dim_2_col  = [None] * H
        iqi_1_col       = [None] * H
        iqi_2_col       = [None] * H

        mag_err_1_mean_col = [None] * H
        mag_err_2_mean_col = [None] * H
        phs_err_1_mean_col = [None] * H
        phs_err_2_mean_col = [None] * H

        mag_err_1_var_col  = [None] * H
        mag_err_2_var_col  = [None] * H
        phs_err_1_var_col  = [None] * H
        phs_err_2_var_col  = [None] * H

        mag_err_1_vec_col  = [None] * H
        mag_err_2_vec_col  = [None] * H
        phs_err_1_vec_col  = [None] * H
        phs_err_2_vec_col  = [None] * H

        short_freq_col = [None] * H
        long_freq_col  = [None] * H
        CFO_col        = [None] * H

        # Known preamble sequence (64 points)
        L_long = np.array([
            0,0,0,0,0,0,1,1,-1,-1,1,1,-1,1,-1,1,1,1,1,1,1,-1,-1,1,1,-1,1,-1,1,1,1,1,
            0,1,-1,-1,1,1,-1,1,-1,1,-1,-1,-1,-1,-1,1,1,-1,-1,1,-1,1,-1,1,1,1,1,0,0,0,0,0
        ], dtype=float)

        # [改4] 52 个有效子载波的位置。12 个空子载波上理想值为 0，没有信号。
        act = np.abs(L_long) > 0

        samples_raw = df[columnName].values
        N = min(len(samples_raw), N_target)

        for idx in range(N):
            complexData = parse_complex_list(samples_raw[idx])
            if complexData.size < 288:
                continue

            # 取 128 + 32 + 128
            S_short = complexData[0:128]     # 8 short preambles (128 samples)
            # S_space = complexData[128:160]  # Protection interval (unused)
            S_long  = complexData[160:288]   # long preamble (128 samples)

            # -------- CFO estimation: short training sequence --------
            a_ST_s_sum = 0+0j
            for i in range(0, 112):  # 0..111
                a_ST_s_sum += np.conj(S_short[i]) * S_short[i + 16]
            a_ST_s = (1.0/16.0) * np.angle(a_ST_s_sum)

            # -------- First carrier frequency offset correction--------
            m = np.arange(len(S_long))
            corrected_1 = S_long * np.exp(-1j * m * a_ST_s)

            # -------- CFO is re-estimated based on the long sequence after one correction. --------
            a_ST_L_sum = 0+0j
            for i in range(0, 64):
                a_ST_L_sum += np.conj(corrected_1[i]) * corrected_1[i + 64]
            a_ST_L = (1.0/64.0) * np.angle(a_ST_L_sum)

            # -------- Second CFO correction --------
            corrected_2 = corrected_1 * np.exp(-1j * m * a_ST_L)

            # ---------------- Frequency domain equalization ----------------
            long1 = corrected_2[0:64]
            long2 = corrected_2[64:128]

            k1 = np.fft.fftshift(np.fft.fft(long1))
            k2 = np.fft.fftshift(np.fft.fft(long2))
            
            EPS = 1e-12

            # 确保复数 NaN
            cNaN = np.nan + 1j * np.nan

            H_est = np.divide(
                0.5 * (k1 + k2),
                L_long,
                out=np.full_like(k1 + k2, cNaN, dtype=np.complex128),
                where=np.isfinite(L_long) & (np.abs(L_long) > EPS)
            )

            # [删1] 与上面的 k1、k2 逐值相同，不再重算
            # X1 = np.fft.fftshift(np.fft.fft(long1))
            # X2 = np.fft.fftshift(np.fft.fft(long2))

            
            Y1 = np.divide(
                k1, H_est,
                out=np.full_like(k1, cNaN, dtype=np.complex128),
                where=np.isfinite(H_est) & (np.abs(H_est) > EPS)
            )

            Y2 = np.divide(
                k2, H_est,
                out=np.full_like(k2, cNaN, dtype=np.complex128),
                where=np.isfinite(H_est) & (np.abs(H_est) > EPS)
            )

            mask1 = np.isnan(Y1.real) | np.isnan(Y1.imag)
            mask2 = np.isnan(Y2.real) | np.isnan(Y2.imag)
            Y1[mask1] = 0
            Y2[mask2] = 0
            # [删2] 复数数组与 None 比较恒为 False，这两行什么也没做
            # Y1 = np.where(Y1 == None, 0, Y1)
            # Y2 = np.where(Y2 == None, 0, Y2)
            

            
            # [改1] 残差减法去掉。均衡后的 Y1、Y2 就是符号本身，
            # 后面的幅度与相位误差公式才对得上。
            # Y1 = Y1 - L_long
            # Y2 = Y2 - L_long

            
            # [改2] 幅度误差改回论文口径 |l| - |l_ideal|
            mag_err_1 = np.abs(Y1) - np.abs(L_long)
            mag_err_2 = np.abs(Y2) - np.abs(L_long)
            # [改3] 相位误差用复数比值取角，避免 2π 回绕
            phs_err_1 = np.angle(Y1 * np.conj(L_long))
            phs_err_2 = np.angle(Y2 * np.conj(L_long))

            # [改4] 只在 52 个有效子载波上统计，12 个空位的 0 不参与
            mag_err_1_mean = float(np.mean(mag_err_1[act]))
            mag_err_2_mean = float(np.mean(mag_err_2[act]))
            phs_err_1_mean = float(np.mean(phs_err_1[act]))
            phs_err_2_mean = float(np.mean(phs_err_2[act]))

            mag_err_1_var  = float(np.var(mag_err_1[act]))
            mag_err_2_var  = float(np.var(mag_err_2[act]))
            phs_err_1_var  = float(np.var(phs_err_1[act]))
            phs_err_2_var  = float(np.var(phs_err_2[act]))

            iq_real_1, iq_imag_1 = np.real(Y1), np.imag(Y1)
            iq_real_2, iq_imag_2 = np.real(Y2), np.imag(Y2)

            # [改5] 分形维数改用 288 点原始前导，在时域算，不做频偏校正
            Y1_frac = frac_dimension(np.real(complexData[0:288]), np.imag(complexData[0:288]), tau)
            # [改5] 两列合一，第二列不再计算
            # Y2_frac = frac_dimension(iq_real_2, iq_imag_2, tau)

            iqi_1 = power(iq_real_1) / max(power(iq_imag_1), 1e-12)
            iqi_2 = power(iq_real_2) / max(power(iq_imag_2), 1e-12)

            iq_preamble_col[idx] = fmt_complex_list(np.r_[Y1, Y2])
            frac_dim_col[idx]    = Y1_frac
            # frac_dim_2_col[idx]  = Y2_frac
            iqi_1_col[idx]       = iqi_1
            iqi_2_col[idx]       = iqi_2

            mag_err_1_mean_col[idx] = mag_err_1_mean
            mag_err_2_mean_col[idx] = mag_err_2_mean
            phs_err_1_mean_col[idx] = phs_err_1_mean
            phs_err_2_mean_col[idx] = phs_err_2_mean

            mag_err_1_var_col[idx]  = mag_err_1_var
            mag_err_2_var_col[idx]  = mag_err_2_var
            phs_err_1_var_col[idx]  = phs_err_1_var
            phs_err_2_var_col[idx]  = phs_err_2_var

            mag_err_1_vec_col[idx]  = '[' + ', '.join(f'{v:g}' for v in mag_err_1.tolist()) + ']'
            mag_err_2_vec_col[idx]  = '[' + ', '.join(f'{v:g}' for v in mag_err_2.tolist()) + ']'
            phs_err_1_vec_col[idx]  = '[' + ', '.join(f'{v:g}' for v in phs_err_1.tolist()) + ']'
            phs_err_2_vec_col[idx]  = '[' + ', '.join(f'{v:g}' for v in phs_err_2.tolist()) + ']'

            short_freq_col[idx] = a_ST_s
            long_freq_col[idx]  = a_ST_L
            CFO_col[idx]        = (a_ST_s + a_ST_L) * 20 * 1_000_000 / (2 * math.pi)

        # Clean up the old column and write to the new column
        for old_col in ('short_freq', 'long_freq', 'frequencyOffset', 'frequencyoffset', 'freq_bench'):
            if old_col in df.columns:
                df.drop(columns=[old_col], inplace=True)

        df['iq_preamble']          = iq_preamble_col
        df['CFO']                  = CFO_col
        df['short_freq']           = short_freq_col
        df['long_freq']            = long_freq_col

        df['frac_dimension']       = frac_dim_col
        # df['frac_dimension_2']     = frac_dim_2_col
        df['iqi_1']                = iqi_1_col
        df['iqi_2']                = iqi_2_col

        df['mag_error_vec_1']      = mag_err_1_vec_col
        df['mag_error_mean_1']     = mag_err_1_mean_col
        df['mag_error_var_1']      = mag_err_1_var_col

        df['mag_error_vec_2']      = mag_err_2_vec_col
        df['mag_error_mean_2']     = mag_err_2_mean_col
        df['mag_error_var_2']      = mag_err_2_var_col

        df['phase_error_vec_1']    = phs_err_1_vec_col
        df['phase_error_mean_1']   = phs_err_1_mean_col
        df['phase_error_var_1']    = phs_err_1_var_col

        df['phase_error_vec_2']    = phs_err_2_vec_col
        df['phase_error_mean_2']   = phs_err_2_mean_col
        df['phase_error_var_2']    = phs_err_2_var_col

        try:
            df.to_csv(dst_path, index=False, encoding='utf-8')
            ok_files.append(src_path.name)
            print(f"Saved file: {dst_path}")
        except Exception as e:
            print(f"[WRITE-ERR] {src_path.name}: {e}")
            error_files.append(src_path.name)

    print(f"[DONE] Saved {len(ok_files)} files to: {dst_dir}")
    if error_files:
        print(f"[WARN] Errors on {len(error_files)} files:")
        for n in error_files:
            print("  -", n)

if __name__ == "__main__":
    main()


# **Display RFF Features**

This section visualizes four representative RF features of two selected devices. 

CFO and phase error are shown as probability distributions, magnitude error as the time-domain waveform of a single frame, and IQI as the constellation of the equalised long training symbols.

In [ ]:
# ======================================================================
# Display RFF Features

import ast
import math
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# ----------------------------------------------------------------------
# 1. Paths and global configuration
# ----------------------------------------------------------------------

# Output directory of the feature computation cell. Point it at
# "/kaggle/working/RFF_DataSet_KF" to use the Kalman-smoothed version.
FEATURE_DIR = "/kaggle/working/RFF_DataSet_with_feature"
OUT_DIR = "/kaggle/working/feature_figures"
os.makedirs(OUT_DIR, exist_ok=True)

# Figure text is all English, and DejaVu Sans ships with Kaggle
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False      # ASCII hyphen for the minus sign

# Fixed device pair, red first and blue second. Each file name is the MAC
# with the colons replaced by underscores.
DEVICES = [
    dict(dev=86, mac="78:21:84:93:56:c8", color="r"),
    dict(dev=2, mac="24:d7:eb:38:c7:fc", color="b"),
]

# The amplitude and constellation panels share one frame, so that the two
# can be read side by side during the tutorial
FRAME_AMP = 139

FS = 20e6                       # IEEE 802.11g baseband sample rate, 20 MHz

# Parameters of the two distribution panels. tick is the x-axis tick interval,
# in the unit of the scaled data. margin is the fractional padding added at
# both ends of the x-axis.
FEATURES = [
    dict(column="CFO", xlabel="CFO/KHz", nbins=170, tick=5.0,
         margin=0.03, scale=1e-3, legend="upper right",
         out="fig1_cfo_distribution.png"),
    dict(column="phase_error_mean_1", xlabel="Phase error", nbins=80, tick=0.2,
         margin=0.04, scale=1.0, legend="upper right",
         out="fig2_phase_error_distribution.png"),
]

# Percentiles used to set the x-range, which trims extreme values at both ends.
# One CFO frame sits more than 8 sigma out, so a min-to-max range would leave
# a large empty gap on the right. Setting 0 and 100 restores the full range.
CLIP_LO, CLIP_HI = 1.0, 99.0

# Whether to save the four panels as separate files as well. By default only
# the 2x2 summary figure is written. Set to True to save four individual
# 300 dpi files, which is handy for slides.
SAVE_INDIVIDUAL = False

# Ideal L-LTF sequence in the frequency domain, 64 real values. The 12 nulls
# sit at indices 0 to 5, the DC at 32, and 59 to 63 at the end.
L_LONG = np.array([
    0, 0, 0, 0, 0, 0, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1,
    1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1, 1, 1, 1,
    0, 1, -1, -1, 1, 1, -1, 1, -1, 1, -1, -1, -1, -1, -1, 1,
    1, -1, -1, 1, -1, 1, -1, 1, 1, 1, 1, 0, 0, 0, 0, 0
], dtype=float)


# ----------------------------------------------------------------------
# 2. Data loading
# ----------------------------------------------------------------------

def feature_path(device):
    """Build the feature file path of one device."""
    return f"{FEATURE_DIR}/{device['mac'].replace(':', '_')}_pre.csv"


COLUMNS = [f["column"] for f in FEATURES]

# Read each device once; the two distribution panels share the data
DATA = {}
for _d in DEVICES:
    _df = pd.read_csv(feature_path(_d), usecols=COLUMNS)
    DATA[_d["mac"]] = {c: pd.to_numeric(_df[c], errors="coerce").values
                       for c in COLUMNS}

print("Feature directory " + FEATURE_DIR)
print(f"Devices {DEVICES[0]['mac']} and {DEVICES[1]['mac']}")
print()


def feature_values(mac, column, scale):
    """
    Take one column, convert it to float, drop NaN, then apply scale.

    The feature computation cell leaves NaN on frames that fail to parse.
    Keeping them makes the percentiles return NaN and ax.set_xlim raises
    ValueError: Axis limits cannot be NaN or Inf.
    """
    v = DATA[mac][column]
    v = v[np.isfinite(v)] * scale
    return v


# The preamble column holds one long string. NumPy right-aligns it, so spaces
# appear between short and long numbers, and a plain split on spaces breaks the
# 288 samples into more than 300 pieces. Strip all whitespace first, then pull
# out the a±bj complexes with a regex, the same way the feature cell does.
COMPLEX_PATTERN = re.compile(
    r'[+-]?(?:\d+(?:\.\d+)?(?:e[+-]?\d+)?)'      # real part
    r'[+-](?:\d+(?:\.\d+)?(?:e[+-]?\d+)?)j',     # imaginary part
    re.IGNORECASE)


def parse_complex_list(cell):
    """Parse one preamble cell into a complex array."""
    if pd.isna(cell):
        return np.array([], dtype=complex)

    s = re.sub(r'\s+', '', str(cell)).replace('i', 'j')
    if not (s.startswith('[') and s.endswith(']')):
        s = f'[{s}]'

    parts = COMPLEX_PATTERN.findall(s)
    if parts:
        try:
            return np.array([complex(p) for p in parts], dtype=complex)
        except Exception:
            pass

    try:
        return np.array(ast.literal_eval(s), dtype=complex)
    except Exception:
        return np.array([], dtype=complex)


def load_frame(device, frame_index):
    """Read the 288 complex samples of one frame from the feature file."""
    df = pd.read_csv(feature_path(device), usecols=["preamble"],
                     nrows=frame_index + 1)
    z = parse_complex_list(df.iloc[frame_index]["preamble"])
    if len(z) != 288:
        raise RuntimeError(
            f"{device['mac']} frame {frame_index} parsed into {len(z)} samples, "
            "expected 288")
    return z


def equalised_symbols(z):
    s_long = z[160:288]                                   # LTS segment, 128 samples

    # Coarse CFO: the STS repeats every 16 samples
    a_sts = (1.0 / 16.0) * np.angle(
        sum(np.conj(z[:128][i]) * z[:128][i + 16] for i in range(112)))

    m = np.arange(128)
    corrected_1 = s_long * np.exp(-1j * m * a_sts)        # first correction

    # Fine CFO: the two halves of the LTS repeat
    a_ltf = (1.0 / 64.0) * np.angle(
        sum(np.conj(corrected_1[i]) * corrected_1[i + 64] for i in range(64)))
    corrected_2 = corrected_1 * np.exp(-1j * m * a_ltf)   # second correction

    k1 = np.fft.fftshift(np.fft.fft(corrected_2[:64]))    # first 64 samples
    k2 = np.fft.fftshift(np.fft.fft(corrected_2[64:]))    # last 64 samples

    # Channel estimate. The null subcarriers are skipped and left at zero.
    valid = np.abs(L_LONG) > 0
    H = np.zeros(64, dtype=complex)
    np.divide(0.5 * (k1 + k2), L_LONG, out=H, where=valid)

    good = valid & (np.abs(H) > 1e-12)
    Y1 = np.zeros(64, dtype=complex)
    Y2 = np.zeros(64, dtype=complex)
    np.divide(k1, H, out=Y1, where=good)
    np.divide(k2, H, out=Y2, where=good)
    return Y1, Y2


# ----------------------------------------------------------------------
# 3. Axis style: inward ticks on all four sides, solid spines, dotted grid
# ----------------------------------------------------------------------

def style_axes(ax, xlabel, ylabel, fs=13, title=None):
    ax.set_axisbelow(True)
    ax.grid(True, which="major", linestyle=":", color="0.35", linewidth=0.9)
    ax.minorticks_on()
    ax.grid(True, which="minor", linestyle=":", color="0.70", linewidth=0.6)
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
        spine.set_color("0.15")
    ax.tick_params(which="both", direction="in", top=True, right=True,
                   length=7, width=1.0, labelsize=fs)
    ax.tick_params(which="minor", length=3.5, width=0.8)
    ax.xaxis.label.set_size(fs + 2)
    ax.yaxis.label.set_size(fs + 2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title, fontsize=fs + 2, pad=10)


def add_mac_legend(ax, loc="upper right", fs=12):
    """Legend labels are the raw MAC addresses."""
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc=loc, fontsize=fs, framealpha=1,
              edgecolor="0.2", borderpad=0.5, handlelength=1.2,
              handleheight=0.9, labelspacing=0.35, fancybox=False)


def nice_ticks(left, right, step):
    """
    Place ticks at multiples of step inside the view limits.

    set_xticks expands the view to fit any tick that falls outside, so a tick
    beyond [left, right] stretches the x-axis and pushes the histogram to one
    side. Keeping every tick inside the limits avoids that.
    """
    first = math.ceil(round(left / step, 9)) * step
    last = math.floor(round(right / step, 9)) * step
    n = int(round((last - first) / step)) + 1
    return np.round(first + step * np.arange(n), 10)


def save(fig, out_name):
    path = f"{OUT_DIR}/{out_name}"
    fig.tight_layout()
    fig.savefig(path, dpi=300)
    plt.show()                  # also render inside the notebook
    plt.close(fig)
    print(f"Saved {path}")


# ----------------------------------------------------------------------
# 4. CFO and phase error: probability distribution of a scalar feature
# ----------------------------------------------------------------------

def draw_histogram(ax, feature, fs=13):
    series = {}
    for d in DEVICES:
        v = feature_values(d["mac"], feature["column"], feature["scale"])
        series[d["mac"]] = (d["color"], v)

    # One shared x-axis, whose ends are the union of both devices' percentiles
    lo = min(np.percentile(v, CLIP_LO) for _, v in series.values())
    hi = max(np.percentile(v, CLIP_HI) for _, v in series.values())
    pad = (hi - lo) * feature["margin"]
    left, right = lo - pad, hi + pad
    edges = np.linspace(left, right, feature["nbins"] + 1)

    for mac, (color, v) in series.items():
        # Weights are 1/N, so each bar height is the probability of its bin,
        # matching the 'probability' normalisation in MATLAB
        weights = np.full(v.size, 1.0 / v.size)
        ax.hist(v, bins=edges, weights=weights, color=color, edgecolor=color,
                linewidth=0.5, label=mac, zorder=3 if color == "b" else 2)

    # Dashed vertical lines at both ends of the x-axis
    for x in (left, right):
        ax.axvline(x, color="0.30", linestyle="--", linewidth=1.0, zorder=1)

    ax.set_xlim(left, right)
    ax.set_xticks(nice_ticks(left, right, feature["tick"]))
    style_axes(ax, feature["xlabel"], "Probability", fs=fs)
    add_mac_legend(ax, loc=feature["legend"], fs=fs - 1)
    return series


# ----------------------------------------------------------------------
# 5. Amplitude: time-domain waveform of one frame, one trace per device
# ----------------------------------------------------------------------

def draw_amplitude(ax, fs=13):
    peak = 0.0
    for d in DEVICES:
        z = load_frame(d, FRAME_AMP)
        t = np.arange(len(z)) / FS                # 288 samples = 14.4 microseconds
        mag = np.abs(z)
        peak = max(peak, float(mag.max()))
        ax.plot(t, mag, color=d["color"], linewidth=0.7, label=d["mac"])
        print(f"    {d['mac']}  frame {FRAME_AMP}  |c| min {mag.min():.4f}  "
              f"max {mag.max():.4f}  RMS {np.sqrt(np.mean(mag ** 2)):.4f}")

    # x-axis in scientific notation, around 10^-5
    formatter = ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((0, 0))
    ax.xaxis.set_major_formatter(formatter)

    # Round the y-limit up to a multiple of 0.05 so the waveform fills the panel
    ymax = math.ceil(peak / 0.05) * 0.05
    ax.set_ylim(0, ymax)
    ax.set_yticks(np.arange(0, ymax + 0.05 * 0.5, 0.05))
    style_axes(ax, "Time", "Magnitude", fs=fs)
    add_mac_legend(ax, fs=fs - 1)


# ----------------------------------------------------------------------
# 6. IQI (I/Q imbalance): constellation of the equalised LTS
#     Red points mark the ideal positions at ±1, blue points the actual
#     symbols Y1 and Y2. Because Y1 + Y2 = 2 * L_LONG, the two sets always
#     sit on opposite sides of the ideal point, and that separation is the
#     I/Q imbalance itself.
# ----------------------------------------------------------------------

def draw_constellation(ax, fs=13, annotate=True, anno_fs=None, title=None):
    anno_fs = fs + 3 if anno_fs is None else anno_fs
    d = DEVICES[0]
    z = load_frame(d, FRAME_AMP)
    Y1, Y2 = equalised_symbols(z)

    # Identity check: Y1 + Y2 should equal 2 * L_LONG exactly
    residual = (Y1 - L_LONG) + (Y2 - L_LONG)
    print(f"    identity check max|Y1+Y2-2L| = {np.max(np.abs(residual)):.3e}")

    # Ideal constellation points: the 52 active subcarriers, all at +1 or -1
    valid = np.abs(L_LONG) > 0
    ideal = L_LONG[valid]
    ax.plot(ideal, np.zeros_like(ideal), "o", color="r", ms=6.5,
            markeredgecolor="darkred", markeredgewidth=0.8, zorder=5)

    # Actual symbols: Y1 and Y2, 64 points each, 128 in total. Of these, 24 land
    # on null subcarriers and read 0, so they stack up on the origin.
    actual = np.r_[Y1, Y2]
    ax.plot(actual.real, actual.imag, "o", color="b", ms=6.5, zorder=3,
            label=d["mac"])

    if annotate:
        # Two arrows, one to the ideal positions and one to the measured cluster
        ax.annotate("Expected I/Q position", xy=(-1.0, 0.03), xycoords="data",
                    xytext=(0.015, 0.905), textcoords="axes fraction",
                    fontsize=anno_fs, color="k", va="top",
                    arrowprops=dict(arrowstyle="->", color="r", lw=1.7,
                                    shrinkA=2, shrinkB=3))
        ax.annotate("", xy=(1.0, 0.03), xytext=(0.30, 0.90),
                    textcoords="axes fraction",
                    arrowprops=dict(arrowstyle="->", color="r", lw=1.7, shrinkA=0))
        ax.annotate("Actual I/Q position", xy=(-0.80, 0.10), xycoords="data",
                    xytext=(0.985, 0.985), textcoords="axes fraction",
                    fontsize=anno_fs, color="k", ha="right", va="top",
                    arrowprops=dict(arrowstyle="->", color="b", lw=1.7,
                                    shrinkA=2, shrinkB=3))
        ax.annotate("", xy=(1.12, 0.18), xytext=(0.80, 0.90),
                    textcoords="axes fraction",
                    arrowprops=dict(arrowstyle="->", color="b", lw=1.7, shrinkA=0))

    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.0, 1.0)
    ax.set_xticks(np.arange(-1.5, 1.6, 0.5))
    ax.set_yticks(np.arange(-1.0, 1.1, 0.5))
    style_axes(ax, "In-phase", "Quadrature", fs=fs, title=title)

    # The device MAC sits in the lower right corner. The two upper corners hold
    # the annotations, and red marks the ideal positions in this panel, so the
    # legend stays clear of both.
    add_mac_legend(ax, loc="lower right", fs=fs - 1)

    print(f"    Y1 real part {Y1.real.min():.4f} to {Y1.real.max():.4f}   "
          f"Y2 real part {Y2.real.min():.4f} to {Y2.real.max():.4f}")
    print(f"    {len(np.unique(np.round(actual, 9)))} distinct positions after "
          "deduplication")


# ----------------------------------------------------------------------
# 7. Per-device statistics, for checking the numbers
# ----------------------------------------------------------------------

for feat in FEATURES:
    print(f"{feat['column']} probability distribution")
    for d in DEVICES:
        v = feature_values(d["mac"], feat["column"], feat["scale"])
        print(f"    {d['mac']}  n={v.size}  min {v.min():.5f}  max {v.max():.5f}  "
              f"mean {v.mean():.5f}  std {v.std():.5f}")
    print()


# ----------------------------------------------------------------------
# 8. Plotting. Only the 2x2 summary figure is written by default
# ----------------------------------------------------------------------

if SAVE_INDIVIDUAL:
    # Individual version, larger, each saved to its own file
    for i, feat in enumerate(FEATURES, 1):
        fig, ax = plt.subplots(figsize=(9.2, 6.9), dpi=300)
        draw_histogram(ax, feat)
        save(fig, feat["out"])

    fig, ax = plt.subplots(figsize=(9.2, 6.9), dpi=300)
    print(f"Amplitude   raw magnitude waveform of frame {FRAME_AMP}")
    draw_amplitude(ax)
    save(fig, "fig3_amplitude_waveform.png")
    print()

    fig, ax = plt.subplots(figsize=(9.6, 6.2), dpi=300)
    print(f"IQI   constellation of frame {FRAME_AMP} for {DEVICES[0]['mac']}")
    draw_constellation(ax, title="Constellation diagram")
    save(fig, "fig4_constellation.png")
    print()

print("2x2 summary figure")
fig, axes = plt.subplots(2, 2, figsize=(15.5, 11.0), dpi=300)
draw_histogram(axes[0, 0], FEATURES[0], fs=15)
draw_histogram(axes[0, 1], FEATURES[1], fs=15)
draw_amplitude(axes[1, 0], fs=15)
# The constellation panel keeps the same two annotations, with a fontsize
# scaled down to fit the quadrant
draw_constellation(axes[1, 1], fs=15, anno_fs=14)
save(fig, "fig_all_features_2x2.png")


# **Kalman Filtering**
(You can skip this step and use the pre-processed dataset named “rffi-kf-feature-iq-wifi-802-11g-2-4g-123-m5stack,” which allows you to directly perform t-SNE visualization and Random Forest classification.)
In this section, the RF features generated in the previous step are refined using Kalman Filtering to improve data smoothness and suppress measurement noise.

In [ ]:
import os
import pandas as pd
import numpy as np
try:
    from pykalman import KalmanFilter
except ImportError:
    !pip install pykalman
    from pykalman import KalmanFilter

# Define the Kalman filter function
def apply_kalman_filter(df, columns):
    filtered_data = df.copy()
    for col in columns:
        if col in df.columns:
            try:
                # Handling NaN and Inf
                col_data = df[col].replace([np.inf, -np.inf], np.nan)
                col_data = col_data.fillna(col_data.median())  # 用中位数填充
                
                # Initialize Kalman filter
                kf = KalmanFilter(initial_state_mean=0, n_dim_obs=1)
                kf = kf.em(col_data.values, n_iter=10)
                
                filtered_col, _ = kf.filter(col_data.values)
                filtered_data[col] = filtered_col
            except Exception as e:
                print(f"Skipping column {col} due to error: {e}")
    return filtered_data

def process_csv_files(input_folder, output_folder, filter_columns):
    for root, _, files in os.walk(input_folder):
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                print(f"Processing file: {file_path}")
                
                df = pd.read_csv(file_path)
                
                # Check if the column to be processed exists.
                cols_to_filter = [col for col in filter_columns if col in df.columns]
                if not cols_to_filter:
                    print(f"No specified columns to process in {file_path}")
                    continue
                
                # Filter the specified column
                filtered_df = apply_kalman_filter(df, cols_to_filter)
                
                relative_path = os.path.relpath(file_path, input_folder)
                output_path = os.path.join(output_folder, relative_path)
                output_dir = os.path.dirname(output_path)
                
                if not os.path.exists(output_dir):
                    os.makedirs(output_dir)
                
                filtered_df.to_csv(output_path, index=False)
                print(f"Saved filtered file to: {output_path}")

# Define the input and output folder paths
input_folders = [
    "/kaggle/working/RFF_DataSet_with_feature"
]
output_folder = "/kaggle/working/RFF_DataSet_KF"

# Columns that need to be filtered
filter_columns = ["CFO", "short_freq", "long_freq", "frac_dimension", "iqi_1", "iqi_2",  "mag_error_mean_1", "mag_error_var_1",  "mag_error_mean_2", "mag_error_var_2",  "phase_error_mean_1", "phase_error_var_1",  "phase_error_mean_2", "phase_error_var_2"]


for folder in input_folders:
    process_csv_files(folder, output_folder, filter_columns)


# **tSNE**
In this section, t-SNE is applied to the extracted features to visualize the distribution before and after Kalman Filtering, highlighting the improvements introduced by the filter.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.manifold import TSNE
import seaborn as sns

plt.rcParams.update({'font.size': 12})
sns.set_context("notebook", font_scale=1.5)

# RFF data without Kalman Filtering  
original_data_path = '/kaggle/working/RFF_DataSet_with_feature'

# Use your self-computed RFF dataset  
#KF_data_path = '/kaggle/working/RFF_DataSet_KF'

# Use our pre-processed RFF dataset  
KF_data_path = '/kaggle/input/rffi-kf-feature-iq-wifi-802-11g-2-4g-123-m5stack'

# Feature list
features = ["CFO", "short_freq", "long_freq","frac_dimension", "iqi_1", "iqi_2",  "mag_error_mean_1", "mag_error_var_1",  "mag_error_mean_2", "mag_error_var_2",  "phase_error_mean_1", "phase_error_var_1",  "phase_error_mean_2", "phase_error_var_2"]


# Initialize the model and encoder
scaler = StandardScaler()
label_encoder = LabelEncoder()

def load_and_preprocess_data(folder_path, label_offset=0):
    """
    从指定文件夹加载CSV数据并预处理
    """
    files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    combined_data = pd.DataFrame()
    
    for file in files:
        file_path = os.path.join(folder_path, file)
        data = pd.read_csv(file_path)
        combined_data = pd.concat([combined_data, data], ignore_index=True)
    
    # Handling missing values
    combined_data = combined_data.dropna(subset=features)
    
    # Processing MAC addresses
    if 'MAC_address' in combined_data.columns:
        combined_data['MAC_label'] = label_encoder.fit_transform(combined_data['MAC_address']) + label_offset

    return combined_data

# Read data
known_data = load_and_preprocess_data(original_data_path)
known_data_kf = load_and_preprocess_data(KF_data_path)

# Randomly select deivec number
rng = np.random.default_rng(seed=42)  # 建立随机数生成器<br/>
fixed_devices = rng.choice(np.arange(1, 124), size=10, replace=False)
print(fixed_devices)

#fixed_devices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# Select samples from the corresponding devices
selected_data = known_data[known_data['MAC_label'].isin(fixed_devices)]
selected_features = selected_data[features]
selected_labels = selected_data['MAC_label']

selected_data_kf = known_data_kf[known_data_kf['MAC_label'].isin(fixed_devices)]
selected_features_kf = selected_data_kf[features]
selected_labels_kf = selected_data_kf['MAC_label']

# Standardized data
selected_features_scaled = scaler.fit_transform(selected_features)

selected_features_scaled_kf = scaler.fit_transform(selected_features_kf)



# t-SNE dimensionality reduction
tsne_2d = TSNE(n_components=2, perplexity=30, random_state=42)
selected_tsne_2d = tsne_2d.fit_transform(selected_features_scaled)

selected_tsne_2d_kf = tsne_2d.fit_transform(selected_features_scaled_kf)



# Color mapping
palette = sns.color_palette("tab10", len(fixed_devices))  
device_color_map = {device: palette[idx] for idx, device in enumerate(fixed_devices)}

# t-SNE 2D Visualization
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=selected_tsne_2d[:, 0], y=selected_tsne_2d[:, 1],
    hue=selected_labels.map(device_color_map), palette=palette, s=50, alpha=0.7, edgecolor='none'
)
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.legend().set_visible(False)
plt.grid()
plt.savefig("tSNE.pdf",bbox_inches="tight")
plt.show()

# t-SNE 2D Visualization under kf dataset
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=selected_tsne_2d_kf[:, 0], y=selected_tsne_2d_kf[:, 1],
    hue=selected_labels_kf.map(device_color_map), palette=palette, s=50, alpha=0.7, edgecolor='none'
)
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.legend().set_visible(False)
plt.grid()
plt.savefig("tSNE_KF.pdf",bbox_inches="tight")
plt.show()


# **Random Forest**
In this section, a Random Forest classifier is constructed, and its identification performance is evaluated on the above two datasets.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import seaborn as sns

plt.rcParams.update({'font.size': 12})
sns.set_context("notebook", font_scale=1.5)



def load_data(folder_path, features, label_column='MAC_address'):
    files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    combined_data = pd.DataFrame()
    for file in files:
        file_path = os.path.join(folder_path, file)
        data = pd.read_csv(file_path)
        combined_data = pd.concat([combined_data, data], ignore_index=True)
    combined_data = combined_data.dropna(subset=features)
    return combined_data



# Calculate the feature correlation matrix (using the Pearson correlation coefficient).
def plot_correlation_matrix(data, features, title="Feature Correlation Matrix (Pearson)"):
    correlation_matrix = data[features].corr(method='pearson')  # 使用 Pearson 相关系数
    plt.figure(figsize=(15, 12))
    sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
    plt.title(title)
    plt.tight_layout()
    plt.savefig("correlation_matrix.pdf",bbox_inches='tight')
    plt.show()


# Feature Importance Analysis Based on Random Forest
def print_feature_importance(model, feature_names):
    importance = model.feature_importances_
    sorted_indices = np.argsort(importance)[::-1]  # sorting in descending order

    print("\n=== Feature Importance Ranking ===")
    for i in sorted_indices:
        print(f"{feature_names[i]:<30s}  Importance: {importance[i]:.6f}")

    importance_df = (
        pd.DataFrame({
            "Feature": np.array(feature_names)[sorted_indices],
            "Importance": importance[sorted_indices]
        })
        .reset_index(drop=True)
    )

    return importance_df



# Random Forest Classification Test Evaluation
def evaluate_model(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    print("\nClassification Report:")
    print(classification_report(y_eval, y_pred))
    print("\nConfusion Matrix:")
    conf = confusion_matrix(y_eval, y_pred)
    np.save("conf.npy",conf)
    plot_cm_heatmap(conf,normalize="row", save_path="cm_row_normalized.png")
    print(conf)
    print(confusion_matrix(y_eval, y_pred))
    print("\nAccuracy Score:")
    print(f"{accuracy_score(y_eval, y_pred):.4f}")

def plot_cm_heatmap(C, normalize="row", title=None, save_path=None, cmap_name="Reds"):
    """
    修改点：
    1) 横纵坐标从 1 开始，并且每隔 2 个显示一次：1,3,5,...
    2) 图更大更清晰：更大的 figsize + 更高 dpi
    3) colorbar 最下面显示 0（显式设置 ticks，且包含 0）
    4) 0 -> 白色；非0 -> colormap（保持你的需求）
    """
    C = np.asarray(C, dtype=float)
    N = C.shape[0]
    assert C.shape == (N, N)

    # ---- normalization ----
    if normalize == "row":
        denom = C.sum(axis=1, keepdims=True)
        Cn = C / np.maximum(denom, 1.0)
        if title is None:
            title = "Confusion Matrix (row-normalized)"
    elif normalize == "col":
        denom = C.sum(axis=0, keepdims=True)
        Cn = C / np.maximum(denom, 1.0)
        if title is None:
            title = "Confusion Matrix (col-normalized)"
    elif normalize == "all":
        denom = C.sum()
        Cn = C / max(denom, 1.0)
        if title is None:
            title = "Confusion Matrix (global-normalized)"
    elif normalize is None:
        Cn = C
        if title is None:
            title = "Confusion Matrix (raw counts)"
    else:
        raise ValueError("normalize must be one of: 'row', 'col', 'all', None")

    np.save("conf_norm.npy", Cn)

    # ---- 0 -> white (mask zeros) ----
    Cn_masked = np.ma.masked_where(Cn == 0, Cn)
    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad(color="white")

    # ---- plot (bigger + sharper) ----
    fig, ax = plt.subplots(figsize=(14, 12), dpi=800)  # 更大+更清晰
    im = ax.imshow(
        Cn_masked,
        aspect="equal",
        cmap=cmap,
        interpolation="nearest"
    )
    ax.set_title(title)

    # ---- ticks: show 1,3,5,... (start from 1) ----
    # tick 位置仍是 0-based index，但显示文字改成 1-based，并且每隔2个显示一次
    tick_pos = np.arange(0, N, 2)         # 0,2,4,...
    tick_lab = (tick_pos + 1).astype(int) # 1,3,5,...

    ax.set_xlabel("Predict label")
    ax.set_ylabel("True label")
    ax.set_xticks(tick_pos)
    ax.set_yticks(tick_pos)
    ax.set_xticklabels(tick_lab)
    ax.set_yticklabels(tick_lab)

    plt.setp(ax.get_xticklabels(), rotation=90, ha="center", fontsize=8)
    plt.setp(ax.get_yticklabels(), fontsize=8)

    # ---- colorbar: force show 0 at bottom ----
    # width 控制宽度；height="100%" 保证和主图同高
    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)  # fraction=宽度, pad=间距

    if normalize in ("row", "col", "all"):
        cbar.set_label("Normalized value")
        # 显式 ticks：保证包含 0
        ticks = np.linspace(0, 1, 6)  # 0,0.2,...,1
        cbar.set_ticks(ticks)
        cbar.set_ticklabels([f"{t:.1f}" for t in ticks])
    else:
        cbar.set_label("Count")
        # counts 也显式包含 0
        vmax = float(np.nanmax(Cn)) if np.nanmax(Cn) > 0 else 1.0
        ticks = np.linspace(0, vmax, 6)
        cbar.set_ticks(ticks)
        cbar.set_ticklabels([f"{t:.0f}" for t in ticks])

    fig.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=800, bbox_inches="tight")  # 保存再更高dpi

    plt.show()
    return Cn
    
def Random_Forest(data_path, features):

    known_data = load_data(data_path, features)

    # Tag Encoding
    #label_encoder = LabelEncoder()
    #known_data['MAC_label'] = label_encoder.fit_transform(known_data['MAC_address'])

    # Separate features and labels
    X_known = known_data[features]
    y_known = known_data['Device Number']

    # Standardization features
    scaler = StandardScaler()
    X_known_scaled = scaler.fit_transform(X_known)

    # Divide the training and evaluation sets
    X_train, X_eval, y_train, y_eval = train_test_split(X_known_scaled, y_known, test_size=0.2, random_state=42)

    # Training a random forest model
    rf_model = RandomForestClassifier(n_estimators=128, random_state=42)
    rf_model.fit(X_train, y_train)

    # Plotting the feature correlation matrix
    plot_correlation_matrix(known_data, features, title="Feature Correlation Matrix (Pearson)")
    # Drawing feature importance
    print_feature_importance(rf_model, features)
    # Evaluate model performance
    evaluate_model(rf_model, X_eval, y_eval)

if __name__ == "__main__":
    # RFF data without Kalman Filtering calculated in this note
    #original_data_path = '/kaggle/working/RFF_DataSet_with_feature'
    
    # Use our pre-processed RFF dataset without Kalman Filtering
    #original_data_path = '/kaggle/input/datasets/zeweiguo/rffi-feature-iq-wifi-802-11g-2-4g-123-m5stack'
    
    # Use your self-computed RFF dataset  
    #KF_data_path = '/kaggle/working/RFF_DataSet_KF'

    # Use our pre-processed RFF dataset after Kalman Filtering
    KF_data_path = '/kaggle/input/rffi-kf-feature-iq-wifi-802-11g-2-4g-123-m5stack'


    
    # Feature selection
    features = ["CFO", "short_freq", "long_freq", "frac_dimension", "iqi_1", "iqi_2",
                "mag_error_mean_1", "mag_error_mean_2", "mag_error_var_1", "mag_error_var_2", "phase_error_mean_1",
                "phase_error_mean_2", "phase_error_var_1", "phase_error_var_2"]
    Random_Forest(original_data_path, features)
    Random_Forest(KF_data_path, features)